# 03 — Baseline Model

Train and evaluate scikit-learn baseline classifiers on MFCC features.

**Steps:**
1. Load MFCC feature vectors from cache (or extract if missing).
2. Train SVM, Random Forest, and Gradient Boosting.
3. Show confusion matrices for each.
4. Bar chart comparing F1 scores across all three.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, classification_report

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.baseline_model import BaselineClassifier

SR = 22050
MANIFEST = os.path.join(PROJECT_ROOT, "data", "manifest.csv")
FEATURE_DIR = os.path.join(PROJECT_ROOT, "data", "features")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models", "baseline")
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

## 1. Load MFCC features

In [ ]:
try:
    from src.features import extract_mfcc_dataset
    X, y = extract_mfcc_dataset(MANIFEST, feature_dir=FEATURE_DIR, sr=SR)
    print(f"Features loaded — X: {X.shape}, y: {y.shape}")
    print(f"  Class 0 (nonfox): {(y == 0).sum()},  Class 1 (fox): {(y == 1).sum()}")
except Exception as e:
    print(f"⚠️  Could not load features: {e}")
    print("   Generating synthetic features for demonstration.")
    rng = np.random.RandomState(42)
    X = rng.randn(100, 80).astype(np.float32)
    y = np.array([0] * 50 + [1] * 50)
    # Make classes slightly separable
    X[y == 1] += 0.5

## 2. Stratified split (70 / 15 / 15)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print(f"Train: {len(y_train)}  Val: {len(y_val)}  Test: {len(y_test)}")

## 3. Train & evaluate all three classifiers

In [ ]:
model_types = ["svm", "random_forest", "gradient_boosting"]
results = {}  # model_type -> {clf, val_metrics, test_metrics}

for mt in model_types:
    print(f"\n{'─' * 40}")
    print(f"  {mt.upper()}")
    print(f"{'─' * 40}")
    clf = BaselineClassifier(model_type=mt)
    clf.train(X_train, y_train)
    val_m = clf.evaluate(X_val, y_val)
    test_m = clf.evaluate(X_test, y_test)
    results[mt] = {"clf": clf, "val": val_m, "test": test_m}
    print(f"  Val  F1={val_m['f1']:.4f}  Acc={val_m['accuracy']:.4f}")
    print(f"  Test F1={test_m['f1']:.4f}  Acc={test_m['accuracy']:.4f}")

## 4. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, mt in zip(axes, model_types):
    clf = results[mt]["clf"]
    y_pred = clf.pipeline.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["nonfox", "fox"],
                yticklabels=["nonfox", "fox"], ax=ax)
    ax.set_title(f"{mt}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

fig.suptitle("Test-set Confusion Matrices", fontsize=13)
fig.tight_layout()
plt.show()

## 5. F1-score comparison bar chart

In [ ]:
f1_vals = [results[mt]["test"]["f1"] for mt in model_types]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(model_types, f1_vals, color=["#4c72b0", "#55a868", "#c44e52"])
for bar, v in zip(bars, f1_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{v:.3f}", ha="center", fontsize=11)
ax.set_ylabel("F1 Score")
ax.set_title("Baseline Classifiers — Test F1 Comparison")
ax.set_ylim(0, 1.1)
fig.tight_layout()
plt.show()

# Identify best
best_mt = model_types[int(np.argmax(f1_vals))]
print(f"\n★ Best baseline: {best_mt} (F1 = {max(f1_vals):.4f})")